# Slice-Aware Scheduling

This notebook demonstrates slice-aware scheduling in Sionna's system-level simulator.
UEs are partitioned into network slices (eMBB and URLLC) using a QFI (QoS Flow Indicator),
and each slice receives a dedicated frequency-resource quota with a per-slice scheduling policy.

**Radio parameters used throughout:**

| Parameter | Value |
|-----------|-------|
| Carrier bandwidth | 100 MHz (FR1) |
| Subcarrier spacing | 30 kHz |
| OFDM symbols per slot | 14 |
| Slot duration | ~0.5 ms (14 × 1/30 kHz ≈ 0.4667 ms without CP) |
| Total PRBs | 273 |
| Total subcarriers (`fft_size`) | 3276 |
| Simulated slots | 100 |

We compare three scenarios:
1. **Baseline** — no slicing, standard `PFSchedulerSUMIMO`
2. **Equal split** — 1638 subcarriers for eMBB (PF) + 1638 for URLLC (priority)
3. **URLLC-favoured** — 655 subcarriers eMBB + 2621 URLLC


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from sionna.sys import (
    SliceAwarePFScheduler,
    validate_slice_config,
    get_slice_prb_ranges,
    build_slice_prb_mask,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == 'cuda' else ""))


## 1  Simulation helpers

Delay-metric helpers that work directly on the `hist` dictionary returned by
`SystemLevelSimulator`.

In [ ]:
def compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean inter-scheduling gap per UE (ms)."""
    is_scheduled = ~np.isnan(hist['harq'])  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            sched_slots = np.where(is_scheduled[:, bs, ue])[0]
            if len(sched_slots) > 1:
                delays[bs, ue] = np.mean(np.diff(sched_slots)) * slot_duration_ms
    return delays


def compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean transmission delay per UE accounting for HARQ retransmissions (ms)."""
    harq = hist['harq']  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            harq_seq = harq[:, bs, ue]
            sched_slots = np.where(~np.isnan(harq_seq))[0]
            if len(sched_slots) == 0:
                continue
            tb_delays, nack_count = [], 0
            for s in sched_slots:
                if harq_seq[s] == 0:
                    nack_count += 1
                elif harq_seq[s] == 1:
                    tb_delays.append((1 + nack_count) * slot_duration_ms)
                    nack_count = 0
            if tb_delays:
                delays[bs, ue] = np.mean(tb_delays)
    return delays


def compute_e2e_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Estimated E2E round-trip delay per UE (ms, symmetric UL/DL assumption)."""
    sched = compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    tx    = compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    one_way = np.nansum([sched, tx], axis=0)
    return 2.0 * one_way, sched, tx

## 2  QFI assignment and slice configurations

**Radio parameters: 100 MHz NR, 30 kHz SCS → 273 PRBs (3276 subcarriers), 14 OFDM symbols/slot, 100 slots.**

- **QFI 1** → eMBB slice (proportional-fair scheduling)
- **QFI 9** → URLLC slice (priority / highest-rate-first scheduling)


In [ ]:
# --- 100 MHz NR carrier, 30 kHz SCS (3GPP TS 38.104 Table 5.3.2-1) ---
# 5 UTs per sector (3 eMBB / 2 URLLC)
num_prbs          = 273           # PRBs (100 MHz NR at 30 kHz SCS)
num_freq_res      = num_prbs * 12 # 3276 subcarriers — matches ResourceGrid.fft_size
num_ofdm_sym      = 14            # OFDM symbols per slot (full NR slot)
num_slots         = 100           # slots to simulate
slot_ms           = 0.5           # nominal slot duration at 30 kHz SCS [ms]
num_ut_per_sector = 5

# QFI assignment: first 3 UEs → eMBB (QFI=1), last 2 → URLLC (QFI=9)
qfi_per_ut = torch.cat([
    torch.full((3,), 1, dtype=torch.int32),
    torch.full((2,), 9, dtype=torch.int32),
])
print("QFI assignment:", qfi_per_ut.tolist())

# --- Slice configurations (must sum to num_freq_res subcarriers) ---
# Each "num_prbs" value is in subcarrier units for scheduler compatibility.
sweep_configs = {
    "equal_split": {
        1: {"num_prbs": num_freq_res // 2,                     "policy": "proportional_fair"},
        9: {"num_prbs": num_freq_res - num_freq_res // 2,      "policy": "priority"},
    },
    "favor_urllc": {
        1: {"num_prbs":  num_freq_res // 5,                     "policy": "proportional_fair"},
        9: {"num_prbs":  num_freq_res - num_freq_res // 5,      "policy": "priority"},
    },
}

for name, cfg in sweep_configs.items():
    validate_slice_config(qfi_per_ut, cfg, num_freq_res)
    print(f"Config '{name}' validated OK "
          f"— subcarriers: {cfg[1]['num_prbs']} eMBB + {cfg[9]['num_prbs']} URLLC")


## 3  PRB mask inspection

Visualise which frequency resources belong to which slice for the equal-split config
(half of subcarriers for eMBB, half for URLLC).


In [ ]:
cfg = sweep_configs["equal_split"]
slice_ranges = get_slice_prb_ranges({q: c['num_prbs'] for q, c in cfg.items()})
print("Subcarrier ranges:", slice_ranges)

mask = build_slice_prb_mask(num_freq_res, qfi_per_ut, slice_ranges)
print(f"PRB mask shape: {mask.shape}  (num_freq_res x num_ut)")

plt.figure(figsize=(12, 3))
plt.imshow(mask.cpu().numpy().T, aspect='auto', origin='lower', cmap='Blues')
plt.xlabel(f"Subcarrier index (0-{num_freq_res - 1})")
plt.ylabel("UE index")
plt.title(f"Subcarrier allocation mask — equal split ({num_prbs} PRBs, 100 MHz, 30 kHz SCS)")
plt.colorbar(label="allowed")
plt.tight_layout()
plt.show()


## 4  Standalone scheduler smoke-test

Sanity check on `SliceAwarePFScheduler` without the full `SystemLevelSimulator`.

In [ ]:
slice_cfg = sweep_configs["equal_split"]

scheduler = SliceAwarePFScheduler(
    qfi_per_ut=qfi_per_ut,
    slice_prb_config=slice_cfg,
    num_ut=num_ut_per_sector,
    num_freq_res=num_freq_res,
    num_ofdm_sym=num_ofdm_sym,
    device=device,
)

rate_last = torch.rand(num_ut_per_sector, device=device) * 100
rate_ach  = torch.rand(num_ofdm_sym, num_freq_res, num_ut_per_sector, device=device) * 100
is_sched  = scheduler(rate_last, rate_ach)  # [sym, prb, ut, streams]

print(f'is_scheduled shape: {is_sched.shape}')

# Verify slice isolation
is_sched_np = is_sched.squeeze(-1).cpu().numpy()  # [sym, prb, ut]
slice_ranges = scheduler.slice_ranges
for qfi_val, (start, end) in slice_ranges.items():
    ut_mask_np = (qfi_per_ut == qfi_val).cpu().numpy()
    out_of_slice = is_sched_np[:, start:end, :][:, :, ~ut_mask_np]
    assert not out_of_slice.any(), f'Slice isolation violated for QFI {qfi_val}!'
print("Slice isolation check: PASSED")

# Verify exactly one UE per PRB
per_prb_sum = is_sched.sum(dim=-2).squeeze(-1)
assert (per_prb_sum == 1).all(), 'Not exactly one UE per PRB!'
print("Single-UE-per-PRB check: PASSED")


## 5  Integration with `SystemLevelSimulator`

This cell runs both a **baseline** and a **slice-aware** simulation using
**100 MHz NR (273 PRBs, `fft_size = 3276` subcarriers), 30 kHz SCS,
14 OFDM symbols per slot, and 100 simulated slots**.

Class definitions and base scenario parameters are loaded automatically
from `End-to-End_Example.ipynb`, and the `ResourceGrid` and `num_slots`
are configured here for the 100 MHz / 14-symbol / 100-slot GPU run.

> **Note on the resource grid**: with 273 PRBs × 12 subcarriers/PRB,
> `fft_size = 3276` subcarriers.  The scheduler's `num_freq_res` parameter
> corresponds to subcarriers (i.e., `ResourceGrid.fft_size`), not PRBs.


In [ ]:
# ==============================================================
# Build SystemLevelSimulator by loading the definitions from
# End-to-End_Example.ipynb and configuring a 100 MHz (273 PRBs,
# 3276 subcarriers), 30 kHz SCS, 14-OFDM-symbol, 100-slot grid.
# ==============================================================

import os as _os, json as _json

# Target radio configuration for this simulation
_target_num_prbs     = 273
_target_fft_size     = _target_num_prbs * 12   # 3276 subcarriers (100 MHz @ 30 kHz SCS)
_target_num_ofdm_sym = 14                      # 14 OFDM symbols/slot
_target_scs          = 30e3                    # 30 kHz subcarrier spacing
_target_num_slots    = 100                     # 100 slots

def _exec_e2e_cells(cell_indices):
    """Execute selected cells from End-to-End_Example.ipynb in this kernel."""
    _nb_path = _os.path.join(_os.path.dirname(_os.path.abspath("__file__")),
                             "End-to-End_Example.ipynb")
    if not _os.path.exists(_nb_path):
        _nb_path = _os.path.join(_os.getcwd(), "End-to-End_Example.ipynb")
    with open(_nb_path) as _f:
        _nb = _json.load(_f)
    _g = globals()
    for _idx in cell_indices:
        _src = "".join(_nb["cells"][_idx]["source"])
        exec(compile(_src, f"E2E_cell_{_idx}", "exec"), _g)

_need_build = True
try:
    _rg = sls.resource_grid
    if (_rg.fft_size == _target_fft_size
            and _rg.num_ofdm_symbols == _target_num_ofdm_sym
            and _rg.subcarrier_spacing == _target_scs):
        _need_build = False
        print(f"Reusing existing sls instance on {sls.device} "
              f"(fft_size={_rg.fft_size}, num_ofdm_symbols={_rg.num_ofdm_symbols}).")
except NameError:
    pass

if _need_build:
    print("Building SystemLevelSimulator (100 MHz, 273 PRBs, 14 OFDM symbols, 100 slots)...")
    # Load class definitions & base scenario parameters from End-to-End_Example.ipynb
    # (skip Cell 22 resource grid & Cell 24 sls instantiation so we override grid first):
    #   4  = sionna import
    #   5  = additional imports
    #   8  = ChannelMatrix class
    #  10  = get_stream_management
    #  12  = get_sinr
    #  14  = estimate_achievable_rate
    #  16  = TensorList + init_result_history + record_results
    #  18  = SystemLevelSimulator class
    #  20  = parameters (direction, scenario, num_rings, num_ut_per_sector, ...)
    #  21  = antenna arrays
    #  29  = num_slots, bler_target, etc.
    _exec_e2e_cells([4, 5, 8, 10, 12, 14, 16, 18, 20, 21, 29])

    # Configure 100 MHz / 30 kHz SCS / 14 OFDM symbols resource grid
    num_ofdm_sym       = _target_num_ofdm_sym
    num_freq_res       = _target_fft_size
    num_subcarriers    = _target_fft_size
    subcarrier_spacing = _target_scs
    resource_grid = ResourceGrid(
        num_ofdm_symbols=num_ofdm_sym,
        fft_size=num_subcarriers,
        subcarrier_spacing=subcarrier_spacing,
        num_tx=num_ut_per_sector,
        num_streams_per_tx=ut_array.num_ant,
    )

    # Instantiate SystemLevelSimulator with the 100 MHz / 14-symbol grid
    _exec_e2e_cells([24])
    num_slots = _target_num_slots
    _fft = sls.resource_grid.fft_size
    print(f"  Built sls on {sls.device}: {sls.num_bs} BSs, "
          f"{sls.num_ut_per_sector} UTs/sector, "
          f"fft_size={_fft} subcarriers ({_fft // 12} PRBs), "
          f"num_ofdm_symbols={sls.resource_grid.num_ofdm_symbols}, "
          f"num_slots={num_slots}")

# Convenience aliases
_num_slots     = _target_num_slots
_alpha_ul      = alpha_ul
_p0_dbm_ul     = p0_dbm_ul
_bler_target   = bler_target
_olla_delta_up = olla_delta_up

# num_freq_res = fft_size (subcarriers), which is what the scheduler expects
_fft = sls.resource_grid.fft_size
num_freq_res = _fft
num_prbs     = _fft // 12

def _to_numpy_hist(h):
    return clean_hist(dict(h))

# ---- Baseline run (standard PF, no slicing) ----
from sionna.sys import PFSchedulerSUMIMO

sls.scheduler = PFSchedulerSUMIMO(
    sls.num_ut_per_sector,
    sls.resource_grid.fft_size,
    sls.resource_grid.num_ofdm_symbols,
    batch_size=[sls.batch_size, sls.num_bs],
    num_streams_per_ut=sls.num_streams_per_ut,
    precision=sls.precision,
    device=sls.device,
)
print('Running baseline simulation...')
hist_baseline = _to_numpy_hist(
    sls(_num_slots, _alpha_ul, _p0_dbm_ul, _bler_target, _olla_delta_up))

_slot_ms  = float(sls.slot_duration * 1e3)
_num_bs   = sls.num_bs
print(f'  slot_duration = {_slot_ms:.4f} ms  (14 OFDM symbols @ 30 kHz SCS)')

# ---- Slice-aware run (equal split scaled to actual num_freq_res) ----
n9 = _fft // 2 + _fft % 2   # URLLC gets the ceiling half
n1 = _fft - n9

# Adjust qfi_per_ut to match actual num_ut_per_sector
_n_ut = sls.num_ut_per_sector
_n_embb   = _n_ut // 2 + _n_ut % 2
_n_urllc  = _n_ut - _n_embb
_qfi = torch.cat([
    torch.full((_n_embb,),  1, dtype=torch.int32),
    torch.full((_n_urllc,), 9, dtype=torch.int32),
])
qfi_per_ut = _qfi   # update for downstream cells

slice_cfg_run = {
    1: {'num_prbs': n1, 'policy': 'proportional_fair'},
    9: {'num_prbs': n9, 'policy': 'priority'},
}
validate_slice_config(qfi_per_ut, slice_cfg_run, _fft)

sls.scheduler = SliceAwarePFScheduler(
    qfi_per_ut=qfi_per_ut,
    slice_prb_config=slice_cfg_run,
    num_ut=sls.num_ut_per_sector,
    num_freq_res=_fft,
    num_ofdm_sym=sls.resource_grid.num_ofdm_symbols,
    batch_size=[sls.batch_size, sls.num_bs],
    num_streams_per_ut=sls.num_streams_per_ut,
    precision=sls.precision,
    device=sls.device,
)
print('Running slice-aware simulation...')
hist_sliced = _to_numpy_hist(
    sls(_num_slots, _alpha_ul, _p0_dbm_ul, _bler_target, _olla_delta_up))

_using_real_hist = True
print('Both simulations complete.')


## 6  KPI computation and comparison

Compute per-slice E2E round-trip delay from `hist_baseline` and `hist_sliced`
produced in Section 5.

In [ ]:
# Compute per-UE delays
e2e_base,   sched_base,   tx_base   = compute_e2e_delay(
    hist_baseline, _num_slots, _num_bs, num_ut_per_sector, _slot_ms)
e2e_sliced, sched_sliced, tx_sliced = compute_e2e_delay(
    hist_sliced, _num_slots, _num_bs, num_ut_per_sector, _slot_ms)

qfi_np = qfi_per_ut.numpy()
slice_names = {1: 'eMBB (QFI=1)', 9: 'URLLC (QFI=9)'}

src_label = 'real simulator' if _using_real_hist else 'synthetic placeholder'
print(f'Results from: {src_label}  |  slot duration = {_slot_ms} ms')
print()
print(f"{'Slice':<20} {'Baseline E2E RTT':>18} {'Slice-aware E2E RTT':>22}")
print("-" * 62)
for qfi_val in sorted(np.unique(qfi_np)):
    ue_mask = qfi_np == qfi_val
    b_mean  = np.nanmean(e2e_base[:, ue_mask])
    s_mean  = np.nanmean(e2e_sliced[:, ue_mask])
    name    = slice_names.get(qfi_val, f'QFI={qfi_val}')
    print(f'{name:<20} {b_mean:>16.2f} ms {s_mean:>20.2f} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
slice_colors = {1: 'steelblue', 9: 'tomato'}

for ax, (scenario_name, e2e) in zip(
        axes, [('Baseline', e2e_base), ('Slice-aware (equal split)', e2e_sliced)]):
    for qfi_val in sorted(np.unique(qfi_np)):
        ue_mask = qfi_np == qfi_val
        vals = e2e[:, ue_mask].ravel()
        vals = vals[~np.isnan(vals)]
        sorted_vals = np.sort(vals)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf,
                color=slice_colors[qfi_val],
                label=slice_names.get(qfi_val, f'QFI={qfi_val}'))
    ax.set_title(scenario_name)
    ax.set_xlabel('E2E round-trip delay (ms)')
    ax.set_ylabel('CDF')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f'E2E RTT CDF per slice  —  100 MHz / 30 kHz SCS / '
    f'{sls.resource_grid.num_ofdm_symbols} sym ({_slot_ms:.2f} ms slot) / {num_prbs} PRBs'
)
plt.tight_layout()
plt.show()


## 7  PRB sweep across configurations

Sweep the URLLC resource fraction from 20 % to 80 % (of the total
frequency resources) and plot per-slice mean E2E delay.  When a real
simulator is available (`_using_real_hist = True`), the simulator is
re-run for each split; otherwise the sweep uses the synthetic
`_make_hist` model.


In [ ]:
urllc_prb_fractions = [0.2, 0.4, 0.5, 0.6, 0.8]
urllc_mean_e2e = []
embb_mean_e2e  = []

for frac in urllc_prb_fractions:
    n_urllc = max(1, round(num_freq_res * frac))
    n_embb  = num_freq_res - n_urllc
    if n_embb < 1:
        continue

    cfg = {
        1: {'num_prbs': n_embb,  'policy': 'proportional_fair'},
        9: {'num_prbs': n_urllc, 'policy': 'priority'},
    }

    if _using_real_hist:
        sls.scheduler = SliceAwarePFScheduler(
            qfi_per_ut=qfi_per_ut,
            slice_prb_config=cfg,
            num_ut=sls.num_ut_per_sector,
            num_freq_res=sls.resource_grid.fft_size,
            num_ofdm_sym=sls.resource_grid.num_ofdm_symbols,
            batch_size=[sls.batch_size, sls.num_bs],
            num_streams_per_ut=sls.num_streams_per_ut,
            precision=sls.precision,
            device=sls.device,
        )
        h_raw = sls(_num_slots, _alpha_ul, _p0_dbm_ul, _bler_target, _olla_delta_up)
        h = _to_numpy_hist(h_raw)
        ns, nb_run = _num_slots, _num_bs
    else:
        p_urllc = 0.1 + 0.8 * frac
        p_embb  = 0.1 + 0.8 * (1 - frac)
        h = _make_hist([p_embb] * 5 + [p_urllc] * 5)
        ns, nb_run = _num_slots, _num_bs

    e2e, _, _ = compute_e2e_delay(h, ns, nb_run, num_ut_per_sector, _slot_ms)
    urllc_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 9]))
    embb_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 1]))
    print(f'frac={frac:.0%}  eMBB={n_embb}  URLLC={n_urllc}  '
          f'->  eMBB E2E={embb_mean_e2e[-1]:.2f} ms  '
          f'URLLC E2E={urllc_mean_e2e[-1]:.2f} ms')

plt.figure(figsize=(7, 4))
plt.plot(urllc_prb_fractions, urllc_mean_e2e, 'o-',  color='tomato',
         label='URLLC (QFI=9)')
plt.plot(urllc_prb_fractions, embb_mean_e2e,  's--', color='steelblue',
         label='eMBB  (QFI=1)')
plt.xlabel(f'URLLC subcarrier fraction  (of {num_freq_res} total, {num_prbs} PRBs)')
plt.ylabel('Mean E2E RTT (ms)')
plt.title(
    f'Per-slice E2E delay vs. resource allocation\n'
    f'100 MHz / 30 kHz SCS / {num_prbs} PRBs / {sls.resource_grid.num_ofdm_symbols} sym ({_slot_ms:.2f} ms slot)'
)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary

| Component | Location |
|-----------|----------|
| `SliceAwarePFScheduler` | `sionna.sys` |
| `build_slice_prb_mask` | `sionna.sys` |
| `validate_slice_config` | `sionna.sys` |
| `get_slice_prb_ranges` | `sionna.sys` |

**Radio configuration:**

| Parameter | Value |
|-----------|-------|
| Carrier bandwidth | 100 MHz (FR1) |
| Subcarrier spacing | 30 kHz |
| OFDM symbols per slot | **14** |
| Slot duration | **~0.5 ms** (`14 / 30 kHz ≈ 0.4667 ms` without CP) |
| Total PRBs | **273** |
| Total subcarriers (`fft_size`) | **3276** |
| Simulated slots | **100** |

> **Note**: The scheduler's `num_freq_res` parameter corresponds to
> subcarriers (= `ResourceGrid.fft_size`), not PRBs.  With 273 PRBs
> × 12 subcarriers/PRB the grid has 3276 frequency resources.
